# 07 — Stacking Ensemble

This notebook builds a stacking ensemble that combines the base learners (RF, XGBoost, LightGBM, SVM, KNN) with a Logistic Regression meta-learner.

**Steps:**
1. Train the stacking ensemble using out-of-fold (OOF) predictions
2. Visualize OOF prediction distributions
3. Compare ensemble performance against individual models

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.base import clone

from src.config import DATASETS, SEED, STACKING_PARAMS
from src.data.loader import load_raw_dataset, get_target_column
from src.data.preprocessor import MediSensePreprocessor
from src.data.splitter import stratified_split
from src.models.base_learners import get_base_learners
from src.models.meta_learner import get_meta_learner
from src.models.stacking import StackingEnsemble
from src.evaluation.metrics import compute_metrics, format_metrics

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## 1. Prepare Data

In [ ]:
def prepare_dataset(dataset_name):
    df = load_raw_dataset(dataset_name)
    target_col = get_target_column(dataset_name)
    
    if dataset_name == 'heart':
        df['ca'] = pd.to_numeric(df['ca'], errors='coerce')
        df['thal'] = pd.to_numeric(df['thal'], errors='coerce')
        df['target'] = (df['target'] > 0).astype(int)
    elif dataset_name == 'liver':
        df['Dataset'] = df['Dataset'].map({1: 1, 2: 0})
    
    train_df, val_df, test_df = stratified_split(df, target_col)
    
    X_train = train_df.drop(columns=[target_col])
    y_train = train_df[target_col].values
    X_test = test_df.drop(columns=[target_col])
    y_test = test_df[target_col].values
    
    preprocessor = MediSensePreprocessor(dataset_name)
    X_train_proc = preprocessor.fit_transform(X_train, y_train)
    X_test_proc = preprocessor.transform(X_test)
    
    return X_train_proc, y_train, X_test_proc, y_test

## 2. Build and Train Stacking Ensemble

In [ ]:
ensemble_results = {}

for ds_name in ['heart', 'diabetes', 'liver']:
    print(f"\n{'=' * 60}")
    print(f"Dataset: {ds_name.upper()}")
    print(f"{'=' * 60}")
    
    X_train, y_train, X_test, y_test = prepare_dataset(ds_name)
    
    # Build ensemble
    base_learners = get_base_learners()
    meta_learner = get_meta_learner()
    ensemble = StackingEnsemble(base_learners, meta_learner)
    
    # Train
    ensemble.fit(X_train, y_train)
    
    # Evaluate ensemble
    y_pred_ens = ensemble.predict(X_test)
    y_prob_ens = ensemble.predict_proba(X_test)[:, 1]
    metrics_ens = compute_metrics(y_test, y_pred_ens, y_prob_ens)
    
    print(f"\n  STACKING ENSEMBLE:")
    print(format_metrics(metrics_ens))
    
    # Evaluate individual models for comparison
    individual_results = []
    for name, learner in get_base_learners():
        fitted = clone(learner)
        fitted.fit(X_train, y_train)
        y_pred_ind = fitted.predict(X_test)
        y_prob_ind = fitted.predict_proba(X_test)[:, 1] if hasattr(fitted, 'predict_proba') else None
        m = compute_metrics(y_test, y_pred_ind, y_prob_ind)
        individual_results.append({'model': name, **m})
    
    individual_results.append({'model': 'stacking', **metrics_ens})
    ensemble_results[ds_name] = individual_results

## 3. OOF Prediction Visualization

We manually generate OOF predictions for the heart dataset to visualize how each base learner contributes.

In [ ]:
# Generate OOF predictions for heart dataset
X_train, y_train, X_test, y_test = prepare_dataset('heart')
base_learners = get_base_learners()
skf = StratifiedKFold(n_splits=STACKING_PARAMS['n_folds'], shuffle=True, random_state=SEED)

n_samples = X_train.shape[0]
n_learners = len(base_learners)
oof_predictions = np.zeros((n_samples, n_learners))

for i, (name, learner) in enumerate(base_learners):
    for train_idx, val_idx in skf.split(X_train, y_train):
        cloned = clone(learner)
        cloned.fit(X_train[train_idx], y_train[train_idx])
        oof_predictions[val_idx, i] = cloned.predict_proba(X_train[val_idx])[:, 1]

oof_df = pd.DataFrame(oof_predictions, columns=[name for name, _ in base_learners])
oof_df['target'] = y_train

print("OOF prediction statistics:")
oof_df.describe()

In [ ]:
# Visualize OOF prediction distributions by target
model_names = [name for name, _ in base_learners]

fig, axes = plt.subplots(1, len(model_names), figsize=(4 * len(model_names), 5))

for i, name in enumerate(model_names):
    ax = axes[i]
    for label in [0, 1]:
        subset = oof_df[oof_df['target'] == label][name]
        sns.kdeplot(subset, ax=ax, label=f'Target={label}', fill=True, alpha=0.4)
    ax.set_title(f'{name.upper()} OOF')
    ax.set_xlabel('Predicted Probability')
    ax.legend()

plt.suptitle('OOF Prediction Distributions by Target — Heart Dataset', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation between OOF predictions of different base learners
fig, ax = plt.subplots(figsize=(8, 6))
oof_corr = oof_df[model_names].corr()
sns.heatmap(oof_corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0.5,
            square=True, ax=ax)
ax.set_title('OOF Prediction Correlation Between Base Learners')
plt.tight_layout()
plt.show()

## 4. Comparison: Stacking vs Individual Models

In [ ]:
for ds_name, results in ensemble_results.items():
    print(f"\n{ds_name.upper()} — Model Comparison:")
    res_df = pd.DataFrame(results)
    display_cols = ['model', 'accuracy', 'f1', 'roc_auc', 'pr_auc']
    cols_present = [c for c in display_cols if c in res_df.columns]
    display(res_df[cols_present].round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for idx, (ds_name, results) in enumerate(ensemble_results.items()):
    ax = axes[idx]
    res_df = pd.DataFrame(results)
    
    colors = ['steelblue'] * (len(res_df) - 1) + ['coral']
    ax.barh(res_df['model'], res_df['f1'], color=colors)
    ax.set_xlabel('F1 Score')
    ax.set_title(f'{ds_name.upper()}')
    ax.set_xlim(0, 1)
    
    for i, v in enumerate(res_df['f1']):
        ax.text(v + 0.01, i, f'{v:.3f}', va='center')

plt.suptitle('F1 Score: Individual Models vs Stacking Ensemble', fontsize=14)
plt.tight_layout()
plt.show()

## Summary

- The stacking ensemble combines predictions from 5 base learners via a Logistic Regression meta-learner.
- OOF predictions show good separation between classes for most base learners.
- Base learner OOF correlations indicate sufficient diversity for effective stacking.
- The stacking ensemble generally matches or exceeds individual model performance, demonstrating the benefit of model combination.